# Supervised Learning Experiments

Run all cells in order. Start this notebook from the repository root or the `docs` directory.
Datasets are read from `docs/`, algorithm implementations are imported from `src/`, and generated files belong in `output/`.


---

# Initialization <a name="2"></a>

---

## Import Libraries

Every import of the notebook is kept in this cell.

In [ ]:
#!pip install matplotlib numpy pandas scikit-learn

In [ ]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.feature_selection import SelectKBest, VarianceThreshold, mutual_info_classif
from sklearn.decomposition import PCA

# Support starting the notebook from the repository root or docs directory.
ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "src").is_dir():
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


## Settings

Paths and shared constants are kept in this class.

In [ ]:
class Settings:
    SEED = 42
    DATA_DIR = ROOT_DIR / "docs"
    OUTPUT_DIR = ROOT_DIR / "output"

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Seed Everything

A fixed seed keeps the results reproducible.

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

The competition files are read here. The `id` column stays as text because some values resemble numbers.

In [ ]:
train_df = pd.read_csv(CFG.DATA_DIR / 'train.csv', dtype={'id': str})
test_df  = pd.read_csv(CFG.DATA_DIR / 'test.csv', dtype={'id': str})

print(f'Train: {train_df.shape}  Test: {test_df.shape}')

---

# Data Understanding <a name="3"></a>

---

## Explore Data
Memahami nya seperti memahami dia


In [ ]:
print(train_df.shape)
train_df.dtypes

In [ ]:
train_df.head()

In [ ]:
train_df.describe()

#### Distribusi Fitur Kategorikal

In [ ]:
train_df["meal_package"].value_counts()


In [ ]:
train_df["room_category"].value_counts()

In [ ]:
train_df["booking_channel"].value_counts()

####  Distribusi is_cancelled

In [ ]:
print(train_df["is_canceled"].value_counts())
print(train_df["is_canceled"].value_counts(normalize=True))

In [ ]:
pd.crosstab(
    train_df["booking_channel"],
    train_df["is_canceled"],
    normalize="index"
)

In [ ]:
pd.crosstab(
    train_df["meal_package"],
    train_df["is_canceled"],
    normalize="index"
)

In [ ]:
train_df["days_before_arrival"].hist(bins=30)
plt.xlabel("Days before arrival")
plt.ylabel("Jumlah pemesanan")
plt.title("Distribusi days_before_arrival")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train_df.boxplot(column="days_before_arrival", ax=axes[0])
axes[0].set_title("Days before arrival")

train_df.boxplot(column="nightly_rate_idr", ax=axes[1])
axes[1].set_title("Nightly rate (IDR)")

plt.tight_layout()
plt.show()

In [ ]:
train_df.groupby("is_canceled")["days_before_arrival"].describe()

## Verify Data Quality

This part covers completeness, consistency, and the validity of values.

In [ ]:
feature_cols = test_df.columns

missing_summary = pd.DataFrame({
    "train": train_df[feature_cols].isna().sum(),
    "test": test_df.isna().sum()
})

missing_summary[missing_summary.sum(axis=1) > 0]

In [ ]:
print("Duplikat baris train:", train_df.duplicated().sum())
print("ID duplikat train:", train_df["id"].duplicated().sum())

print("Duplikat baris test:", test_df.duplicated().sum())
print("ID duplikat test:", test_df["id"].duplicated().sum())


In [ ]:
nonnegative_cols = [
    "num_adults",
    "num_children",
    "weekend_nights",
    "weekday_nights",
    "days_before_arrival",
    "prior_cancellations",
    "prior_completed_bookings",
    "nightly_rate_idr",
    "special_requests",
]

train_df[nonnegative_cols].lt(0).sum()

In [ ]:
print("num_adults = 0:", (train_df["num_adults"] == 0).sum())
print("arrival_month di luar 1–12:", (~train_df["arrival_month"].between(1, 12)).sum())
print("arrival_day di luar 1–31:", (~train_df["arrival_day"].between(1, 31)).sum())

In [ ]:
train_features = train_df.drop(columns="is_canceled")

print("Kolom train dan test sama:", train_features.columns.equals(test_df.columns))

for col in ["meal_package", "room_category", "booking_channel"]:
    unseen = set(test_df[col].dropna().unique()) - set(train_df[col].dropna().unique())
    print(f"{col}, kategori baru di test:", unseen)

---

# Training and Validation Split <a name="4"></a>

---

The split comes before data preparation so that validation data stays unseen. Only `train.csv` is split, and `test.csv` serves the Kaggle submission alone.

In [ ]:
train_set, val_set = train_test_split(
    train_df,
    test_size=0.20,
    random_state=CFG.SEED,
    stratify=train_df["is_canceled"],
)

print(f"Training set:   {train_set.shape}")
print(f"Validation set: {val_set.shape}")
print("Proporsi target training:")
print(train_set["is_canceled"].value_counts(normalize=True).sort_index())
print("Proporsi target validation:")
print(val_set["is_canceled"].value_counts(normalize=True).sort_index())


---

# Data Preparation <a name="5"></a>

---

## Select Data

In [ ]:
TARGET_COL = "is_canceled"
ID_COL = "id"
feature_cols = [col for col in test_df.columns if col != ID_COL]

test_ids = test_df[ID_COL].copy()

X_train_raw = train_set[feature_cols].copy()
y_train = train_set[TARGET_COL].astype(int).copy()
X_val_raw = val_set[feature_cols].copy()
y_val = val_set[TARGET_COL].astype(int).copy()
X_test_raw = test_df[feature_cols].copy()

print("Jumlah fitur:", len(feature_cols))
print("Train:", X_train_raw.shape, y_train.shape)
print("Validation:", X_val_raw.shape, y_val.shape)
print("Test:", X_test_raw.shape)


## Clean Data

Cleaning raises data quality to the level the chosen models need.

### Missing Values

In [ ]:
train_zero_adults = X_train_raw["num_adults"].eq(0)
val_zero_adults = X_val_raw["num_adults"].eq(0)
test_zero_adults = X_test_raw["num_adults"].eq(0)

print("num_adults = 0 pada train:", int(train_zero_adults.sum()))
print("num_adults = 0 pada validation:", int(val_zero_adults.sum()))
print("num_adults = 0 pada test:", int(test_zero_adults.sum()))

X_train_raw.loc[train_zero_adults, "num_adults"] = np.nan
X_val_raw.loc[val_zero_adults, "num_adults"] = np.nan
X_test_raw.loc[test_zero_adults, "num_adults"] = np.nan

numeric_cols = X_train_raw.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train_raw.select_dtypes(exclude="number").columns.tolist()

numeric_fill_values = X_train_raw[numeric_cols].median()
categorical_fill_values = X_train_raw[categorical_cols].mode(dropna=True).iloc[0]

def impute_missing_values(data: pd.DataFrame) -> pd.DataFrame:
    result = data.copy()
    result[numeric_cols] = result[numeric_cols].fillna(numeric_fill_values)
    result[categorical_cols] = result[categorical_cols].fillna(categorical_fill_values)
    return result

X_train_imputed = impute_missing_values(X_train_raw)
X_val_imputed = impute_missing_values(X_val_raw)
X_test_imputed = impute_missing_values(X_test_raw)

print("Missing value setelah imputasi (train):", int(X_train_imputed.isna().sum().sum()))
print("Missing value setelah imputasi (validation):", int(X_val_imputed.isna().sum().sum()))
print("Missing value setelah imputasi (test):", int(X_test_imputed.isna().sum().sum()))

### Duplicates

In [ ]:
print("Duplikat train:", int(train_set.duplicated().sum()))
print("Duplikat validation:", int(val_set.duplicated().sum()))
print("Duplikat test:", int(test_df.duplicated().sum()))

X_train_clean = X_train_imputed.copy()
X_val_clean = X_val_imputed.copy()
X_test_clean = X_test_imputed.copy()


### Outliers

In [ ]:
outlier_cols = [
    "num_children", "weekend_nights", "weekday_nights",
    "days_before_arrival", "prior_cancellations",
    "prior_completed_bookings", "nightly_rate_idr", "special_requests",
]
outlier_counts = {}

for col in outlier_cols:
    values = X_train_raw[col].dropna()
    if col == "nightly_rate_idr":
        values = values[values > 0]
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    source = X_train_raw[col]
    eligible = source.notna()
    if col == "nightly_rate_idr":
        eligible &= source > 0
    outlier_counts[col] = int((eligible & ((source < lower) | (source > upper))).sum())

pd.Series(outlier_counts, name="kandidat_outlier_IQR")


### Data Errors

In [ ]:
negative_counts = train_set[numeric_cols].lt(0).sum()

binary_cols = ["needs_parking", "is_returning_guest", "is_canceled"]
invalid_binary_counts = (
    (~train_set[binary_cols].isin([0, 1]) & train_set[binary_cols].notna()).sum()
)

arrival_dates = pd.to_datetime(
    train_set[["arrival_year", "arrival_month", "arrival_day"]]
    .rename(columns={
        "arrival_year": "year",
        "arrival_month": "month",
        "arrival_day": "day",
    }),
    errors="coerce",
)
invalid_arrival_dates = int(arrival_dates.isna().sum())

print("Nilai numerik negatif:", negative_counts[negative_counts > 0].to_dict())
print("Nilai biner di luar 0/1:", invalid_binary_counts[invalid_binary_counts > 0].to_dict())
print("Tanggal kedatangan tidak valid:", invalid_arrival_dates)

## Data Transformation

### Feature Scaling

This part covers normalization to the range 0 to 1 and standardization.

In [ ]:
scaled_numeric_cols = X_train_clean.select_dtypes(include="number").columns.tolist()
scaler = StandardScaler()

X_train_scaled = X_train_clean.copy()
X_val_scaled = X_val_clean.copy()
X_test_scaled = X_test_clean.copy()

X_train_scaled[scaled_numeric_cols] = scaler.fit_transform(X_train_clean[scaled_numeric_cols])
X_val_scaled[scaled_numeric_cols] = scaler.transform(X_val_clean[scaled_numeric_cols])
X_test_scaled[scaled_numeric_cols] = scaler.transform(X_test_clean[scaled_numeric_cols])

print("Jumlah fitur numerik yang distandardisasi:", len(scaled_numeric_cols))


### Encoding

This part covers the numeric representation of categorical attributes.

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(X_train_scaled[categorical_cols])

encoded_feature_names = encoder.get_feature_names_out(categorical_cols)

def combine_scaled_and_encoded(data: pd.DataFrame) -> pd.DataFrame:
    numeric_part = data[scaled_numeric_cols].copy()
    categorical_array = encoder.transform(data[categorical_cols])
    categorical_part = pd.DataFrame(
        categorical_array,
        columns=encoded_feature_names,
        index=data.index,
    )
    return pd.concat([numeric_part, categorical_part], axis=1)

X_train_encoded = combine_scaled_and_encoded(X_train_scaled)
X_val_encoded = combine_scaled_and_encoded(X_val_scaled)
X_test_encoded = combine_scaled_and_encoded(X_test_scaled)

print("Bentuk fitur train:", X_train_encoded.shape)
print("Bentuk fitur validation:", X_val_encoded.shape)
print("Bentuk fitur test:", X_test_encoded.shape)


## Feature Selection

Remove constant features, then rank the remaining encoded features using mutual
information with the target. Keep up to 15 features as an experimental subset;
this is a starting point, not a claim that 15 is optimal. Compare this subset with
all features during model validation before choosing the final representation.

Both steps are fitted on the original training split only, before oversampling.
Validation and Kaggle test data only receive the fitted transformations.

Reference: https://scikit-learn.org/stable/modules/feature_selection.html

In [ ]:
constant_filter = VarianceThreshold(threshold=0.0)
constant_filter.fit(X_train_encoded)
nonconstant_cols = X_train_encoded.columns[constant_filter.get_support()].tolist()
removed_constant_cols = X_train_encoded.columns[~constant_filter.get_support()].tolist()

# One-hot columns and binary flags are discrete, even after scaling.
discrete_features = np.array([
    col not in scaled_numeric_cols or col in ["needs_parking", "is_returning_guest"]
    for col in nonconstant_cols
])


def score_features(X, y):
    values = np.array(X, dtype=float, copy=True)
    # Give discrete features integer codes after their earlier standardization.
    for i, is_discrete in enumerate(discrete_features):
        if is_discrete:
            _, values[:, i] = np.unique(values[:, i], return_inverse=True)
    return mutual_info_classif(
        values, y, discrete_features=discrete_features, random_state=CFG.SEED
    )


SELECT_K = min(15, len(nonconstant_cols))
selector = SelectKBest(score_func=score_features, k=SELECT_K)
selector.fit(X_train_encoded[nonconstant_cols], y_train)
selected_cols = np.array(nonconstant_cols)[selector.get_support()].tolist()

feature_scores = pd.Series(
    selector.scores_, index=nonconstant_cols, name="mutual_information"
).sort_values(ascending=False)
print("Removed constant features:", removed_constant_cols)
print("Feature scores:")
print(feature_scores)
print("Selected features:", selected_cols)

X_train_selected = X_train_encoded[selected_cols].copy()
X_val_selected = X_val_encoded[selected_cols].copy()
X_test_selected = X_test_encoded[selected_cols].copy()

feature_scores.sort_values().plot.barh(figsize=(8, 8))
plt.xlabel("Mutual information with is_canceled")
plt.title("Feature relevance on the training split")
plt.tight_layout()
plt.show()

## Dimensionality Reduction

PCA is a separate experiment on the selected features. Standardize its input,
including the one-hot columns, so PCA does not depend on differences in feature
scales. Retain at least 95% of the training variance. This percentage describes
feature variance, not classification accuracy.

Fit the additional scaler and PCA only on training data, before oversampling.
Keep the full, selected, and PCA representations for later validation comparisons.
PCA components are numeric and should not be passed to the categorical part of
mixed Naive Bayes. Its original categorical input is preserved separately below.

Reference: https://scikit-learn.org/stable/modules/decomposition.html#pca

In [ ]:
pca_scaler = StandardScaler()
pca_train = pca_scaler.fit_transform(X_train_selected)
pca_val = pca_scaler.transform(X_val_selected)
pca_test = pca_scaler.transform(X_test_selected)

pca = PCA(n_components=0.95, svd_solver="full")
X_train_pca = pd.DataFrame(pca.fit_transform(pca_train), index=X_train_selected.index)
X_val_pca = pd.DataFrame(pca.transform(pca_val), index=X_val_selected.index)
X_test_pca = pd.DataFrame(pca.transform(pca_test), index=X_test_selected.index)

print("Selected feature count:", len(selected_cols))
print("PCA component count:", pca.n_components_)
print("Retained variance:", pca.explained_variance_ratio_.sum())

cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance, marker="o")
plt.axhline(0.95, linestyle="--", label="95% variance")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA on the selected training features")
plt.legend()
plt.tight_layout()
plt.show()

### Class Balance

In [ ]:
class_balance = pd.DataFrame({
    "jumlah": y_train.value_counts().sort_index(),
    "proporsi": y_train.value_counts(normalize=True).sort_index(),
})
class_balance.index.name = TARGET_COL
print("Distribusi sebelum oversampling:")
print(class_balance)

majority_indices = y_train[y_train == 0].index
minority_indices = y_train[y_train == 1].sample(
    n=len(majority_indices),
    replace=True,
    random_state=CFG.SEED,
).index
balanced_indices = majority_indices.append(minority_indices)
balanced_indices = pd.Series(balanced_indices).sample(
    frac=1,
    random_state=CFG.SEED,
).to_numpy()

X_train_final = X_train_encoded.loc[balanced_indices].reset_index(drop=True)
y_train_final = y_train.loc[balanced_indices].reset_index(drop=True)
X_val_final = X_val_encoded.copy()
X_test_final = X_test_encoded.copy()

print("Distribusi training setelah oversampling:")
print(y_train_final.value_counts().sort_index())


# Apply the same sampled row indices to each representation.
X_train_selected_final = X_train_selected.loc[balanced_indices].reset_index(drop=True)
X_val_selected_final = X_val_selected.copy()
X_test_selected_final = X_test_selected.copy()

X_train_pca_final = X_train_pca.loc[balanced_indices].reset_index(drop=True)
X_val_pca_final = X_val_pca.copy()
X_test_pca_final = X_test_pca.copy()

# Preserve pre-one-hot data for mixed Naive Bayes.
nb_categorical_cols = categorical_cols + ["needs_parking", "is_returning_guest"]
nb_numeric_cols = [col for col in numeric_cols if col not in nb_categorical_cols]
X_train_nb = X_train_clean.loc[balanced_indices].reset_index(drop=True)
X_val_nb = X_val_clean.copy()
X_test_nb = X_test_clean.copy()
y_train_nb = y_train.loc[balanced_indices].reset_index(drop=True)
y_val_nb = y_val.copy()

print("Full training features:", X_train_final.shape)
print("Selected training features:", X_train_selected_final.shape)
print("PCA training components:", X_train_pca_final.shape)
print("Mixed NB training features:", X_train_nb.shape)


## Format Data

Formatting produces the structure the models read, such as a numeric matrix.

In [ ]:
X_train = X_train_final.to_numpy(dtype=float)
X_val = X_val_final.to_numpy(dtype=float)
X_test = X_test_final.to_numpy(dtype=float)
y_train = y_train_final.to_numpy(dtype=int)
y_val = y_val.to_numpy(dtype=int)

print("X_train, y_train:", X_train.shape, y_train.shape)
print("X_val, y_val:", X_val.shape, y_val.shape)
print("X_test:", X_test.shape)

## Available Model Inputs

- Existing `X_train`, `X_val`, `X_test`: all encoded features as NumPy arrays.
- `X_train_selected_final`, `X_val_selected_final`, `X_test_selected_final`:
  the selected encoded features as DataFrames.
- `X_train_pca_final`, `X_val_pca_final`, `X_test_pca_final`:
  numeric PCA components as DataFrames.
- The three representations above use the final `y_train` and `y_val` arrays.
- `X_train_nb`, `X_val_nb`, `X_test_nb`: cleaned features before scaling and
  one-hot encoding, with their own `y_train_nb` and `y_val_nb` labels.
  Pass `nb_numeric_cols` and `nb_categorical_cols` to mixed Naive Bayes.

All training representations use the same oversampling indices. Validation and
Kaggle test rows are neither resampled nor removed. The mixed NB baseline retains
all original features; the selection experiment currently applies to encoded data.